[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S01_python_fundamentos.ipynb)

# Sesión 01 · Fundamentos de Python

**Módulo 1: Python** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Moverte en Colab: ejecutar celdas, escribir comentarios y leer un mensaje de error.
2. Guardar datos en variables y actualizarlas (reasignación).
3. Distinguir `int`, `float`, `str` y `bool`, y convertir de un tipo a otro.
4. Hacer cálculos respetando la precedencia, incluidos `//` y `%` con números negativos.
5. Limpiar un texto y extraer sus partes con índices, slicing, métodos y f-strings.

## 📋 Qué debes saber antes
Nada. Esta es la primera sesión.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos de práctica y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión y las funciones que revisan tus respuestas.
import hashlib
import math
import re

import numpy as np

rng = np.random.default_rng(42)

# ---------- Datos de práctica: ventas de tiendas ----------
_TIENDAS = ["Miraflores", "San Isidro", "Surco", "Barranco", "Lince"]
_PRODUCTOS = ["Polo Algodón", "Jean Clásico", "Casaca Denim", "Zapatillas Urbanas", "Gorra Bordada"]

stock_inicial = int(rng.integers(40, 80))
vendidas_manana = int(rng.integers(5, 20))
vendidas_tarde = int(rng.integers(5, 20))
reposicion = int(rng.integers(10, 30))

unidades_txt = str(int(rng.integers(2, 12)))
precio_txt = f"{rng.uniform(20, 150):.2f}"

precio_lista = round(float(rng.uniform(50, 300)), 2)
descuento_pct = int(rng.choice([10, 15, 20, 25]))
ventas_hoy = round(float(rng.uniform(1000, 5000)), 2)
crecimiento_pct = int(rng.integers(3, 12))

unidades_pedido = int(rng.integers(40, 100))
por_caja = int(rng.choice([6, 8, 12]))

_t = _TIENDAS[int(rng.integers(len(_TIENDAS)))]
_p = _PRODUCTOS[int(rng.integers(len(_PRODUCTOS)))]
codigo_venta = f"LIM-{_t[:3].upper()}-2026-{int(rng.integers(1, 99999)):05d}"
_precio_coma = f"{rng.uniform(15, 250):.2f}".replace(".", ",")
registro_crudo = f"   2026-09-25;{_t.lower()};{_p};{int(rng.integers(1, 10))};{_precio_coma}  \n"

# ---------- Datos de práctica: movimientos bancarios ----------
saldo_inicial = round(float(rng.uniform(800, 3000)), 2)
movimiento = f"  25/09/2026 | retiro cajero   |-{int(rng.integers(5, 60)) * 10:.2f}| pen  "

# ---------- Nivel pro ----------
monto_grande = round(float(rng.uniform(1_000_000, 9_000_000)), 3)
tasa_morosidad = float(rng.uniform(0.01, 0.2))

# Copia privada: los verificadores no dependen de lo que cambies arriba.
_D = {k: globals()[k] for k in [
    "stock_inicial", "vendidas_manana", "vendidas_tarde", "reposicion", "unidades_txt",
    "precio_txt", "precio_lista", "descuento_pct", "ventas_hoy", "crecimiento_pct",
    "unidades_pedido", "por_caja", "codigo_venta", "registro_crudo", "saldo_inicial",
    "movimiento", "monto_grande", "tasa_morosidad",
]}

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro la variable `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and type(valor) is not tipo:
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {tipo.__name__}.")
            return _FALTA
        return valor

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le falta {cuantos} al final." if n == 1 else f"`{nombre}` está incompleto: le faltan {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def check_ejercicio_1():
    r = _Revision("Ejercicio 1")
    p = r.var("precio_gorra")
    c = r.var("cantidad_gorras")
    t = r.var("total_gorras")
    if p is not _FALTA and c is not _FALTA:
        r.ok("`precio_gorra` y `cantidad_gorras` existen.")
    if t is not _FALTA:
        if not isinstance(t, (int, float)) or isinstance(t, bool):
            r.mal(f"`total_gorras` debería ser un número y es {type(t).__name__}.")
        elif _cerca(t, sum([25.5] * 4)):
            r.ok("`total_gorras` tiene el importe correcto.")
        else:
            r.mal(f"`total_gorras` vale {t}, que no es precio por cantidad. ¿Quedó algún nombre mal escrito?")
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2")
    s = r.var("stock", int)
    if s is not _FALTA:
        esperado = _D["stock_inicial"]
        for cambio in (-_D["vendidas_manana"], -_D["vendidas_tarde"], _D["reposicion"]):
            esperado += cambio
        if s == esperado:
            r.ok("`stock` refleja los tres movimientos del día.")
        elif s == _D["stock_inicial"]:
            r.mal("`stock` sigue igual que `stock_inicial`: ¿reasignaste la variable en cada paso?")
        else:
            r.mal(f"`stock` vale {s}, que no es el stock final. Revisa qué movimientos restan, cuáles suman y que no falte ninguno.")
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    txt_u, txt_p = _D["unidades_txt"], _D["precio_txt"]
    u_ref = sum(int(ch) * 10 ** i for i, ch in enumerate(reversed(txt_u)))
    entero, dec = txt_p.split(".")
    p_ref = int(entero) + int(dec) / 10 ** len(dec)
    u = r.var("unidades_num", int)
    if u is not _FALTA:
        r.ok("`unidades_num` es correcto.") if u == u_ref else r.mal(f"`unidades_num` vale {u}; debería salir de convertir `unidades_txt`.")
    p = r.var("precio_num", float)
    if p is not _FALTA:
        r.ok("`precio_num` es correcto.") if _cerca(p, p_ref) else r.mal(f"`precio_num` vale {p}; debería salir de convertir `precio_txt`.")
    i = r.var("importe", float)
    if i is not _FALTA:
        r.ok("`importe` es correcto.") if _cerca(i, u_ref * p_ref, 1e-6) else r.mal(f"`importe` vale {i}; revisa que sea unidades por precio.")
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_1": "471f35af4d44003f2a65b0c1c13eafa17a3cf85679e0d6d44d25818fc84bebfb",
        "pred_2": "2ef0d86cc94a0960e013359fc270f1a50efdf896ddf7102af42fcbe173d4ed07",
        "pred_3": "471f35af4d44003f2a65b0c1c13eafa17a3cf85679e0d6d44d25818fc84bebfb",
        "pred_4": "4bfc84ac8fd1e690b2309147b1a4cddd6d7a2714fa6aa501a4f3d95ced8e3158",
        "pred_5": "504fd320c035cf419fcfb0f0cbe7f3b478d10f78fda40022eeb35a86b23bce08",
        "pred_6": "98f1c15b97f0aab4934b54ee33b6a9fd148f07ee171d3618e796e5ba5b2cbf3d",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    pl, d = _D["precio_lista"], _D["descuento_pct"]
    rebajado = pl - pl * d / 100
    ref = rebajado + rebajado * 18 / 100
    v = r.var("precio_final")
    if v is not _FALTA:
        if not isinstance(v, float):
            r.mal(f"`precio_final` debería ser float y es {type(v).__name__}.")
        elif abs(v - ref) <= 0.011:
            if _dos_decimales(v):
                r.ok("`precio_final` es correcto y está redondeado.")
            else:
                r.mal("El valor de `precio_final` va bien, pero falta redondearlo a 2 decimales con `round()`.")
        elif abs(v - rebajado) <= 0.011:
            r.mal("`precio_final` tiene el descuento pero le falta el IGV.")
        elif abs(v - pl * 1.18) <= 0.011:
            r.mal("`precio_final` tiene el IGV pero le falta el descuento.")
        else:
            r.mal(f"`precio_final` vale {v}. Revisa la precedencia: ¿los paréntesis encierran lo que debe calcularse primero?")
    vh, c = _D["ventas_hoy"], _D["crecimiento_pct"]
    ref = vh
    for _ in range(3):
        ref = ref + ref * c / 100
    r.fin()
    r2 = _Revision("Ejercicio 4 · Parte B")
    v = r2.var("proyeccion")
    if v is not _FALTA:
        if not isinstance(v, float):
            r2.mal(f"`proyeccion` debería ser float y es {type(v).__name__}.")
        elif abs(v - ref) <= 0.011:
            if _dos_decimales(v):
                r2.ok("`proyeccion` es correcta y está redondeada.")
            else:
                r2.mal("El valor de `proyeccion` va bien, pero falta redondearlo a 2 decimales.")
        elif abs(v - vh * (1 + 3 * c / 100)) <= 0.011:
            r2.mal("Sumaste el porcentaje tres veces sobre las ventas de hoy; cada mes debe crecer sobre el mes anterior.")
        else:
            r2.mal(f"`proyeccion` vale {v}. Revisa dónde actúa `**` y dónde van los paréntesis.")
    r2.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    resto, llenas = _D["unidades_pedido"], 0
    while resto >= _D["por_caja"]:
        resto -= _D["por_caja"]
        llenas += 1
    for nombre, ref in (("cajas", llenas), ("sueltas", resto)):
        v = r.var(nombre)
        if v is _FALTA:
            continue
        if type(v) is float:
            r.mal(f"`{nombre}` es float: ¿usaste `/`? Aquí necesitas un resultado entero.")
        elif type(v) is not int:
            r.mal(f"`{nombre}` debería ser int y es {type(v).__name__}.")
        elif v == ref:
            r.ok(f"`{nombre}` es correcto.")
        else:
            r.mal(f"`{nombre}` vale {v}, no coincide. Comprueba que `por_caja * cajas + sueltas` dé `unidades_pedido`.")
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    q, m = globals().get("pred_div"), globals().get("pred_mod")
    r.predicciones({
        "pred_div": "43e8af843ea9cc7217627910ac8fc5af1dc7b254eabfbc530806c0c5f9a0b65c",
        "pred_mod": "4bfc84ac8fd1e690b2309147b1a4cddd6d7a2714fa6aa501a4f3d95ced8e3158",
        "pred_menor": "7a20ad6fbd72c40ec9ce42fcce75b2afd01e02ef19725a1956c8b5a8b66eeefc",
    })
    if q == -3:
        print("   💡 En `pred_div` redondeaste hacia cero; `//` redondea hacia abajo.")
    if type(q) is int and type(m) is int and 5 * q + m != -17:
        r.mal("Tus `pred_div` y `pred_mod` no cumplen `a == b * (a // b) + (a % b)` con a = -17 y b = 5.")
    r.fin()


def check_ejercicio_6():
    r = _Revision("Ejercicio 6")
    partes = _D["codigo_venta"].split("-")
    v = r.var("ciudad", str)
    if v is not _FALTA:
        r.ok("`ciudad` es correcta.") if v == partes[0] else r.mal(f"`ciudad` vale {v!r}; deberían ser los 3 primeros caracteres.")
    v = r.var("anio", int)
    if v is not _FALTA:
        if v == int(partes[2]):
            r.ok("`anio` es correcto.")
        elif len(str(v)) != 4:
            r.mal(f"`anio` tiene {len(str(v))} dígitos y un año tiene 4. Recuerda que en `[inicio:fin]` el `fin` no se incluye.")
        else:
            r.mal(f"`anio` vale {v}; revisa las posiciones del slicing.")
    v = r.var("correlativo", str)
    if v is not _FALTA:
        if v == partes[3]:
            r.ok("`correlativo` es correcto.")
        elif len(v) != 5:
            r.mal(f"`correlativo` tiene {len(v)} caracteres y se esperaban 5.")
        else:
            r.mal(f"`correlativo` vale {v!r}; deberían ser los 5 últimos caracteres.")
    v = r.var("largo", int)
    if v is not _FALTA:
        ref = sum(1 for _ in _D["codigo_venta"])
        r.ok("`largo` es correcto.") if v == ref else r.mal(f"`largo` vale {v}; cuenta los caracteres de todo el código, guiones incluidos.")
    r.fin()


def _registro():
    m = re.match(r"\s*([\d-]+);([^;]+);([^;]+);(\d+);(\d+),(\d+)\s*$", _D["registro_crudo"])
    return {
        "tienda": m.group(2).upper(),
        "producto": m.group(3).lower(),
        "unidades": int(m.group(4)),
        "precio": int(m.group(5)) + int(m.group(6)) / 100,
    }


def check_ejercicio_7():
    r = _Revision("Ejercicio 7 · Parte A")
    ref = _registro()
    for nombre, como in (("tienda", "en MAYÚSCULAS"), ("producto", "en minúsculas")):
        v = r.var(nombre, str)
        if v is _FALTA or not r.texto_limpio(nombre, v):
            continue
        if v == ref[nombre]:
            r.ok(f"`{nombre}` es correcto.")
        elif v.lower() == ref[nombre].lower():
            r.mal(f"`{nombre}` tiene el texto correcto pero no está {como}.")
        else:
            r.mal(f"`{nombre}` vale {v!r}; revisa qué posición de la lista tomaste.")
    v = r.var("unidades", int)
    if v is not _FALTA:
        r.ok("`unidades` es correcto.") if v == ref["unidades"] else r.mal(f"`unidades` vale {v}; revisa qué posición tomaste.")
    v = r.var("precio", float)
    if v is not _FALTA:
        r.ok("`precio` es correcto.") if _cerca(v, ref["precio"]) else r.mal(f"`precio` vale {v}; revisa la posición y el cambio de coma por punto.")
    r.fin()
    r = _Revision("Ejercicio 7 · Parte B")
    r.predicciones({
        "pred_len_vacio": "25af31a57c2047d854d189042b0ecfb66843c4c19d3dd9ce1403e0be3826a240",
        "pred_partes": "d024f6472cd1381eeddb28a59daaff16528770c069c615713e06c20d6b4498fd",
        "pred_partes_vacio": "d024f6472cd1381eeddb28a59daaff16528770c069c615713e06c20d6b4498fd",
    })
    r.fin()


def check_ejercicio_8():
    r = _Revision("Ejercicio 8")
    ref = _registro()
    esperado = "%s | %s | %d x S/ %.2f = S/ %.2f" % (
        ref["tienda"], ref["producto"], ref["unidades"], ref["precio"], ref["unidades"] * ref["precio"])
    v = r.var("ticket", str)
    if v is not _FALTA:
        if v == esperado:
            r.ok("`ticket` tiene el formato exacto.")
        else:
            _primera_diferencia(r, "ticket", v, esperado)
            print("   Si el ejercicio 7 no está superado, corrígelo primero.")
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    m = re.match(r"\s*(\d\d)/(\d\d)/(\d{4})\s*\|\s*(.*?)\s*\|\s*(-?\d+)\.(\d\d)\s*\|\s*(\w+)\s*$", _D["movimiento"])
    f_ref = f"{m.group(1)}/{m.group(2)}/{m.group(3)}"
    c_ref = m.group(4).upper()
    monto_ref = -(int(m.group(5)[1:]) + int(m.group(6)) / 100)
    mon_ref = m.group(7).upper()
    for nombre, ref in (("fecha", f_ref), ("concepto", c_ref), ("moneda", mon_ref)):
        v = r.var(nombre, str)
        if v is _FALTA or not r.texto_limpio(nombre, v):
            continue
        if v == ref:
            r.ok(f"`{nombre}` es correcto.")
        elif v.lower() == ref.lower():
            r.mal(f"`{nombre}` tiene el texto correcto pero no está en el formato pedido (mayúsculas).")
        else:
            r.mal(f"`{nombre}` vale {v!r}; revisa qué parte del movimiento tomaste.")
    v = r.var("monto", float)
    if v is not _FALTA:
        if _cerca(v, monto_ref):
            r.ok("`monto` es correcto.")
        elif _cerca(v, -monto_ref):
            r.mal("`monto` debería ser negativo: es un retiro.")
        else:
            r.mal(f"`monto` vale {v}; revisa qué parte del movimiento convertiste.")
    v = r.var("mes", int)
    if v is not _FALTA:
        if v == int(m.group(2)):
            r.ok("`mes` es correcto.")
        elif not 1 <= v <= 12:
            r.mal(f"`mes` vale {v}, que no es un mes válido (1 a 12).")
        else:
            r.mal(f"`mes` vale {v}; en `dd/mm/aaaa` el mes es la parte del medio.")
    v = r.var("saldo_final")
    saldo_ref = _D["saldo_inicial"] + monto_ref
    if v is not _FALTA:
        if not isinstance(v, float):
            r.mal(f"`saldo_final` debería ser float y es {type(v).__name__}.")
        elif abs(v - saldo_ref) <= 0.011:
            r.ok("`saldo_final` es correcto.") if _dos_decimales(v) else r.mal("`saldo_final` va bien pero falta redondearlo a 2 decimales.")
        elif abs(v - (_D["saldo_inicial"] - monto_ref)) <= 0.011:
            r.mal("`saldo_final` es mayor que `saldo_inicial`, pero un retiro reduce el saldo. Revisa el signo.")
        else:
            r.mal(f"`saldo_final` vale {v}; debería ser el saldo inicial después del movimiento.")
    restante = int(m.group(5)[1:])
    refs = {}
    for billete in (100, 50, 10):
        refs[billete] = 0
        while restante >= billete:
            restante -= billete
            refs[billete] += 1
    billetes = {}
    for billete in (100, 50, 10):
        nombre = f"billetes_{billete}"
        v = r.var(nombre)
        if v is _FALTA:
            continue
        if type(v) is not int:
            r.mal(f"`{nombre}` debería ser int y es {type(v).__name__}. Convierte con `int()` antes de usar `//` y `%`.")
            continue
        billetes[billete] = v
        r.ok(f"`{nombre}` es correcto.") if v == refs[billete] else r.mal(f"`{nombre}` vale {v}, no coincide.")
    if len(billetes) == 3:
        suma = sum(b * n for b, n in billetes.items())
        if suma != int(m.group(5)[1:]):
            r.mal(f"Tus billetes suman S/ {suma}, que no es el importe retirado.")
    v = r.var("resumen", str)
    if v is not _FALTA:
        esperado = "%s | %s | S/ %.2f | saldo: S/ %.2f" % (f_ref, c_ref, monto_ref, round(saldo_ref, 2))
        if v == esperado:
            r.ok("`resumen` tiene el formato exacto.")
        else:
            _primera_diferencia(r, "resumen", v, esperado)
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    ent, dec = ("%.2f" % _D["monto_grande"]).split(".")
    grupos = []
    while ent:
        grupos.insert(0, ent[-3:])
        ent = ent[:-3]
    ref_miles = "S/ " + ",".join(grupos) + "." + dec
    ref_pct = "%.1f%%" % (_D["tasa_morosidad"] * 100)
    ref_inv = "".join(reversed(_D["codigo_venta"]))
    for nombre, ref in (("texto_miles", ref_miles), ("texto_pct", ref_pct), ("codigo_invertido", ref_inv)):
        v = r.var(nombre, str)
        if v is _FALTA:
            continue
        if v == ref:
            r.ok(f"`{nombre}` es correcto.")
        else:
            _primera_diferencia(r, nombre, v, ref)
    r.fin()


print("✅ Setup listo. Datos generados y verificadores cargados.")


### 📦 Tus datos de hoy
El setup creó estas variables. Los valores se generan con una semilla fija, así que siempre salen iguales. Los textos se muestran entre comillas para que veas los espacios sobrantes; `\n` es un salto de línea.

In [ ]:
print("🏪 Ventas de tiendas")
print("stock_inicial   =", stock_inicial)
print("vendidas_manana =", vendidas_manana)
print("vendidas_tarde  =", vendidas_tarde)
print("reposicion      =", reposicion)
print("unidades_txt    =", repr(unidades_txt))
print("precio_txt      =", repr(precio_txt))
print("precio_lista    =", precio_lista)
print("descuento_pct   =", descuento_pct)
print("ventas_hoy      =", ventas_hoy)
print("crecimiento_pct =", crecimiento_pct)
print("unidades_pedido =", unidades_pedido)
print("por_caja        =", por_caja)
print("codigo_venta    =", repr(codigo_venta))
print("registro_crudo  =", repr(registro_crudo))
print()
print("🏦 Movimiento bancario")
print("saldo_inicial   =", saldo_inicial)
print("movimiento      =", repr(movimiento))

---
## 1. Colab, `print` y cómo leer un error

### 📘 Concepto
Un notebook tiene dos tipos de celdas:
- **Texto**, como esta: explicaciones.
- **Código**: Python que ejecutas con **Shift + Enter** o con el botón ▶️.

`print()` muestra valores en pantalla; si le pasas varios separados por comas, los separa con un espacio. Todo lo que va después de `#` es un **comentario**: Python lo ignora y sirve para explicar el código. Si la última línea de una celda es una expresión, Colab muestra su valor aunque no uses `print`.

In [ ]:
# Esto es un comentario: Python no lo ejecuta.
print("Hola, analista")
print("Ventas del día:", 1520, "soles")   # varios valores separados por comas
1520 + 380                                # la última expresión se muestra sola

**Leer un error.** Cuando algo falla, Python muestra un *traceback*. Léelo **de abajo hacia arriba**:
1. La **última línea** dice el tipo de error y el motivo.
2. La flecha `---->` señala la línea que falló.

Esto es lo que verías al ejecutar `print(ventas_ayer)` sin haber creado esa variable:
```
NameError                                 Traceback (most recent call last)
----> 1 print(ventas_ayer)

NameError: name 'ventas_ayer' is not defined
```

Errores frecuentes:
| Error | Qué suele significar |
|---|---|
| `NameError` | usas un nombre que no existe (o está mal escrito) |
| `SyntaxError` | el código está mal escrito: un paréntesis o una comilla sin cerrar |
| `TypeError` | operas con tipos que no combinan, como texto + número |
| `ValueError` | el valor no se puede convertir, como `int("hola")` |

Pruébalo: quita el `#` de la línea de abajo, ejecuta, lee el error y vuelve a poner el `#`.

In [ ]:
# print(ventas_ayer)

### ✍️ Tu turno · Ejercicio 1: arregla los errores
Este código tiene **dos** errores:
```python
precio_gorra = 25.5
cantidad_gorras = 4
total_gorras = precio_gorra * cantidad_gorra
print("Total:", total_gorras
```
1. Cópialo en la celda de abajo y ejecútalo. Lee el error, corrígelo y vuelve a ejecutar hasta que aparezca el total.
2. No cambies los nombres `precio_gorra`, `cantidad_gorras` ni `total_gorras`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Python solo te muestra un error a la vez. Arregla el primero que aparezca, ejecuta de nuevo y verás el siguiente.
</details>

<details><summary>💡 Pista 2</summary>

Uno de los errores es de sintaxis (mira los paréntesis de la última línea). El otro es un `NameError`: compara, letra por letra, el nombre que Python dice que no existe con los nombres que sí creaste.
</details>

---
## 2. Variables y reasignación

### 📘 Concepto
Una **variable** es un nombre que guarda un valor. Se crea con `=`, que se lee "guarda en": el nombre va a la izquierda y el valor a la derecha.

- Usa minúsculas y guiones bajos: `ventas_lima`. Sin espacios y sin empezar con número; evita tildes y ñ.
- **Reasignar** es guardar un valor nuevo en la misma variable; el anterior se pierde.
- Puedes usar el valor actual para calcular el nuevo: `x = x + 1`. Atajos: `x += 1` y `x -= 1`.

In [ ]:
caja = 500          # la caja empieza con 500 soles
print("Inicio:", caja)

caja = caja + 120   # entra una venta de 120
print("Tras la venta:", caja)

caja -= 45          # sale un pago de 45 (atajo de caja = caja - 45)
print("Tras el pago:", caja)

### ✍️ Tu turno · Ejercicio 2: el stock del día
La tienda empieza el día con `stock_inicial` unidades. Durante el día:
1. En la mañana se venden `vendidas_manana` unidades.
2. En la tarde se venden `vendidas_tarde` unidades.
3. Llega una reposición de `reposicion` unidades.

Crea una variable `stock` igual a `stock_inicial` y **reasígnala una vez por cada movimiento**, en ese orden, con un `print` después de cada paso. Usa las variables; no escribas los números a mano.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Empieza con `stock = stock_inicial`. En cada línea siguiente, `stock` aparece a los dos lados del `=` (o usas un atajo como `-=`).
</details>

<details><summary>💡 Pista 2</summary>

Las ventas sacan unidades del almacén y la reposición las agrega. Son tres líneas de reasignación, una por movimiento.
</details>

---
## 3. Tipos de datos y conversión

### 📘 Concepto
| Tipo | Qué guarda | Ejemplos |
|---|---|---|
| `int` | números enteros | `12`, `-3`, `0` |
| `float` | números con decimales | `29.9`, `-0.5`, `3.0` |
| `str` | texto, entre comillas | `"Lince"`, `'42'` |
| `bool` | verdadero o falso | `True`, `False` |

`type(valor)` te dice el tipo. Para convertir usa `int()`, `float()`, `str()` y `bool()`.

Detalles que importan:
- `"42"` es texto aunque parezca número: `"42" + "1"` da `"421"`.
- `/` siempre da `float`, aunque la división sea exacta.
- Si una operación mezcla `int` y `float`, el resultado es `float`.
- `bool()` da `False` solo con valores "vacíos": el número `0`, el `0.0` y el texto vacío `""`. Todo lo demás da `True`.
- No puedes unir texto y número con `+`: convierte antes el número con `str()`.

In [ ]:
unidades_ej = 3
precio_ej = 19.9
tienda_ej = "Surco"
abierta_ej = True
print(type(unidades_ej), type(precio_ej), type(tienda_ej), type(abierta_ej))

texto_ej = "15"
print(texto_ej + "5")          # une textos
print(int(texto_ej) + 5)       # convierte y suma
print(float("19.90") * 2)
print("Unidades: " + str(unidades_ej))   # str() para unir con texto
print(bool(0), bool(15), bool("Surco"))

# int("19.90")   # ValueError: ese texto tiene decimales; para eso está float()

### ✍️ Tu turno · Ejercicio 3: convertir y predecir
**Parte A.** Las variables `unidades_txt` y `precio_txt` llegaron como texto desde un sistema de caja.
1. Crea `unidades_num` con `unidades_txt` convertido a entero.
2. Crea `precio_num` con `precio_txt` convertido a decimal.
3. Crea `importe`: unidades por precio.

**Parte B.** Predice **sin ejecutar** y guarda tu respuesta en cada variable:

| Variable | Pregunta | Cómo escribir la respuesta |
|---|---|---|
| `pred_1` | ¿de qué tipo es `10 / 2`? | el nombre del tipo como texto, por ejemplo `"str"` |
| `pred_2` | ¿de qué tipo es `int("8") * 2`? | el nombre del tipo como texto |
| `pred_3` | ¿de qué tipo es `3 + 2.0`? | el nombre del tipo como texto |
| `pred_4` | ¿qué valor da `int(3.99)`? | un número |
| `pred_5` | ¿qué valor da `bool("0")`? | `True` o `False`, sin comillas |
| `pred_6` | ¿qué valor da `bool("")`? | `True` o `False`, sin comillas |

Cuando verifiques, comprueba cada expresión ejecutándola en una celda nueva.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Parte A: `int()` y `float()` reciben un texto y devuelven un número. Parte B: relee la lista "Detalles que importan".
</details>

<details><summary>💡 Pista 2</summary>

Para `pred_4`: `int()` con un número decimal no redondea al más cercano; piensa qué hace con los decimales. Para `pred_5` y `pred_6`: a `bool()` no le importa lo que diga el texto, solo si está vacío o no.
</details>

---
## 4. Operadores y precedencia

### 📘 Concepto
| Operador | Qué hace | Ejemplo | Resultado |
|---|---|---|---|
| `+` `-` `*` | suma, resta, multiplicación | `7 * 3` | `21` |
| `/` | división | `7 / 2` | `3.5` |
| `//` | división entera | `7 // 2` | `3` |
| `%` | resto de la división entera | `7 % 2` | `1` |
| `**` | potencia | `2 ** 3` | `8` |

**Precedencia**, de mayor a menor: paréntesis → `**` → `*` `/` `//` `%` → `+` `-`. Con igual precedencia se evalúa de izquierda a derecha. Si dudas, usa paréntesis.

`round(x, 2)` redondea `x` a 2 decimales.

In [ ]:
print(2 + 3 * 4)         # primero la multiplicación: 14
print((2 + 3) * 4)       # los paréntesis mandan: 20

base_ej = 50
print(base_ej + base_ej * 18 / 100)     # 50 más el 18 % de 50
print((base_ej + base_ej) * 18 / 100)   # otra cosa: el 18 % de 100

print(2 * 3 ** 2)        # la potencia va primero: 2 * 9
print((2 * 3) ** 2)      # 6 al cuadrado

print(round(10 / 3, 2))

### ✍️ Tu turno · Ejercicio 4: precio final y proyección
**Parte A.** Un producto cuesta `precio_lista` soles sin IGV. Tiene un descuento de `descuento_pct` por ciento y, sobre el precio ya rebajado, se cobra un IGV del 18 %. Calcula `precio_final` en **una sola expresión** y redondéalo a 2 decimales.

**Parte B.** Las ventas de hoy son `ventas_hoy`. Si crecen `crecimiento_pct` por ciento cada mes, siempre sobre el mes anterior, ¿cuánto se venderá dentro de 3 meses? Guárdalo en `proyeccion`, redondeado a 2 decimales. Usa `**`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Parte A: si te enredas, calcula por pasos en variables auxiliares y, cuando funcione, júntalo en una sola expresión. Parte B: crecer un 10 % es multiplicar por 1.10.
</details>

<details><summary>💡 Pista 2</summary>

Un 15 % de descuento significa que pagas el 85 % del precio. ¿Cómo escribes ese "85 %" con `descuento_pct`? Revisa que los paréntesis rodeen cada factor completo. En la parte B, el factor de un mes se aplica tres veces seguidas.
</details>

---
## 5. `//` y `%`, también con negativos

### 📘 Concepto
`a // b` dice cuántas veces cabe `b` en `a`, redondeando **hacia abajo** (hacia menos infinito, no hacia cero). `a % b` es lo que sobra. Siempre se cumple:

```
a == b * (a // b) + (a % b)
```

Con `b` positivo, `a % b` siempre queda entre `0` y `b - 1`, aunque `a` sea negativo. Por eso, con negativos, `//` da un número "más bajo" de lo que quizá esperabas. Dividir entre `0` con `/`, `//` o `%` da `ZeroDivisionError`.

In [ ]:
print(23 // 5, 23 % 5)             # caben 4 cajas de 5 y sobran 3
print(-7 // 2, -7 % 2)             # -3.5 redondeado hacia abajo es -4; sobra 1
print(2 * (-7 // 2) + (-7 % 2))    # comprobación: vuelve a dar -7

### ✍️ Tu turno · Ejercicio 5: cajas y restos
**Parte A.** Hay que despachar `unidades_pedido` unidades en cajas de `por_caja` unidades. Calcula `cajas` (cajas completas) y `sueltas` (unidades que no llenan una caja). Usa las variables.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta |
|---|---|
| `pred_div` | ¿qué valor da `-17 // 5`? |
| `pred_mod` | ¿qué valor da `-17 % 5`? |
| `pred_menor` | ¿qué valor da `4 % 9`? (el número es menor que el divisor) |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Parte A: una variable sale con `//` y la otra con `%`. Parte B: calcula primero `-17 / 5` con decimales y redondea hacia abajo en la recta numérica.
</details>

<details><summary>💡 Pista 2</summary>

Con `pred_div` en la mano, despeja el resto de `a == b * (a // b) + (a % b)`. Para `pred_menor`: ¿cuántas veces cabe 9 en 4, y cuánto sobra?
</details>

---
## 6. Textos: índices, slicing y `len`

### 📘 Concepto
Un texto es una secuencia de caracteres. Cada uno tiene una posición (**índice**) que empieza en **0**. Los índices negativos cuentan desde el final: `-1` es el último.

| Carácter | S | U | R | C | O |
|---|---|---|---|---|---|
| Índice | 0 | 1 | 2 | 3 | 4 |
| Índice negativo | -5 | -4 | -3 | -2 | -1 |

- `texto[i]`: el carácter en la posición `i`.
- `texto[inicio:fin]`: desde `inicio` hasta **antes** de `fin`. Sin `inicio`, empieza al principio; sin `fin`, llega al final.
- `len(texto)`: cuántos caracteres tiene. Los espacios también cuentan.

In [ ]:
sku = "POL-AZU-M"
print(sku[0], sku[-1])      # primer y último carácter
print(sku[0:3])             # posiciones 0, 1 y 2
print(sku[4:7])             # posiciones 4, 5 y 6
print(sku[:3], sku[-1:])    # sin inicio / sin fin
print(len(sku))

### ✍️ Tu turno · Ejercicio 6: descomponer un código
`codigo_venta` tiene la forma `CIUDAD-TIENDA-AÑO-CORRELATIVO`. Usa índices y slicing (no uses `split`) para crear:
1. `ciudad`: los 3 primeros caracteres.
2. `anio`: el año, convertido a `int`.
3. `correlativo`: los 5 últimos caracteres, como texto (los ceros de la izquierda importan). Usa índices negativos.
4. `largo`: la cantidad de caracteres del código.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_6()

<details><summary>💡 Pista 1</summary>

Imprime `codigo_venta` y escribe debajo, a mano, el índice de cada carácter. Los guiones también ocupan una posición.
</details>

<details><summary>💡 Pista 2</summary>

El año empieza justo después del segundo guion y ocupa 4 caracteres; `fin` es la posición siguiente al último que quieres. Para el final del texto, `[-5:]` significa "desde el quinto contando desde el final hasta el final".
</details>

---
## 7. Textos: métodos para limpiar y cortar

### 📘 Concepto
Los métodos se escriben `texto.metodo()` y **devuelven un texto nuevo**: el original no cambia, así que guarda el resultado si lo necesitas.

| Método | Qué hace |
|---|---|
| `.upper()` / `.lower()` | pasa a mayúsculas / minúsculas |
| `.strip()` | quita espacios y saltos de línea al inicio y al final |
| `.replace(viejo, nuevo)` | reemplaza todas las apariciones de `viejo` |
| `.split(sep)` | corta el texto en cada `sep` y devuelve una **lista** de partes |

Una **lista** es una secuencia de valores entre `[ ]`. Por ahora basta con saber que accedes a sus elementos por índice, igual que en un texto; la verás a fondo en la sesión 2. Puedes encadenar métodos: `texto.strip().upper()`.

In [ ]:
linea = "   Jean Clásico  \n"
print(repr(linea))              # repr muestra los espacios y el \n
limpia = linea.strip()
print(repr(limpia))
print(limpia.upper(), "|", limpia.lower())
print(limpia.replace("Clásico", "Slim"))

datos_ej = "Surco|Polo|3"
partes_ej = datos_ej.split("|")
print(partes_ej, len(partes_ej))
print(partes_ej[0].upper(), partes_ej[-1])

### ✍️ Tu turno · Ejercicio 7: limpiar un registro de ventas
`registro_crudo` es una línea exportada de un sistema de ventas: tiene espacios sobrantes, un salto de línea al final y campos separados por `;` en este orden: `fecha;tienda;producto;unidades;precio`. El precio usa **coma** decimal.

**Parte A.** Crea:
1. `tienda`: en MAYÚSCULAS.
2. `producto`: en minúsculas.
3. `unidades`: como `int`.
4. `precio`: como `float` (primero cambia la coma por un punto).

Ninguno debe tener espacios al inicio ni al final.

**Parte B · casos borde.** Predice **sin ejecutar**:

| Variable | Pregunta |
|---|---|
| `pred_len_vacio` | ¿qué valor da `len("    ".strip())`? |
| `pred_partes` | ¿qué valor da `len("sin separador".split(";"))`? (no hay ningún `;`) |
| `pred_partes_vacio` | ¿qué valor da `len("".split(";"))`? |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_7()

<details><summary>💡 Pista 1</summary>

Primero limpia la línea entera y córtala por `;`; guarda la lista en una variable. Luego toma cada campo por su posición.
</details>

<details><summary>💡 Pista 2</summary>

La tienda es el campo en la posición 1, no en la 0. Para el precio encadena: el campo → `.replace(...)` → `float(...)`. En la parte B, prueba mentalmente: si no hay nada que cortar, ¿`split` devuelve cero partes o una?
</details>

---
## 8. f-strings

### 📘 Concepto
Un **f-string** es un texto con una `f` delante en el que metes valores entre llaves: `f"Hola {nombre}"`. Dentro de las llaves puedes poner expresiones, y después de `:` puedes darles formato:
- `{x:.2f}`: con 2 decimales.
- `{x:.1%}`: como porcentaje con 1 decimal.

In [ ]:
tienda_ej = "Barranco"
ventas_ej = 3450.5
meta_ej = 4000
print(f"{tienda_ej} vendió S/ {ventas_ej:.2f}")
print(f"Faltan S/ {meta_ej - ventas_ej:.2f} para la meta")
print(f"Avance: {ventas_ej / meta_ej:.1%}")

### ✍️ Tu turno · Ejercicio 8: el ticket
Con las variables del ejercicio 7, crea `ticket` con este formato exacto:

`TIENDA | producto | UNIDADES x S/ PRECIO = S/ TOTAL`

- `PRECIO` y `TOTAL` con 2 decimales; `TOTAL` es unidades por precio.
- Entre campos: espacio, barra vertical, espacio.
- Ejemplo con otros datos: `CENTRO | jean clásico | 2 x S/ 45.50 = S/ 91.00`

Luego haz `print(ticket)`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_8()

<details><summary>💡 Pista 1</summary>

Copia el ejemplo dentro de `f"..."` y reemplaza cada dato por su variable entre llaves.
</details>

<details><summary>💡 Pista 2</summary>

El total no necesita variable propia: puedes escribir la multiplicación dentro de las llaves y darle formato con `:.2f`.
</details>

---
## 🏋️ Reto final: un movimiento bancario
Tu banco exporta cada movimiento como una línea de texto. La variable `movimiento` tiene los campos `fecha | concepto | monto | moneda`, con espacios de sobra; el monto es negativo porque es un retiro. `saldo_inicial` es el saldo antes del movimiento.

Crea estas variables:
1. `fecha` (texto `dd/mm/aaaa`), `concepto` (en MAYÚSCULAS), `monto` (`float`, negativo) y `moneda` (en MAYÚSCULAS). Ninguno con espacios a los lados.
2. `mes`: el mes de la fecha, como `int`.
3. `saldo_final`: el saldo después del movimiento, redondeado a 2 decimales.
4. El cajero entregó el dinero con la menor cantidad posible de billetes de 100, 50 y 10. Calcula `billetes_100`, `billetes_50` y `billetes_10` (todos `int`) usando `//` y `%`.
5. `resumen` con este formato exacto (montos con 2 decimales):

   `FECHA | CONCEPTO | S/ MONTO | saldo: S/ SALDO_FINAL`

   Ejemplo con otros datos: `01/01/2026 | PAGO SERVICIO | S/ -80.00 | saldo: S/ 1420.50`

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Divide el trabajo: limpia y corta la línea una vez, y después limpia cada campo por separado. Imprime cada variable con `repr()` para ver si quedaron espacios.
</details>

<details><summary>💡 Pista 2</summary>

Para los billetes, trabaja con el importe retirado como número **positivo y entero** (piensa en qué hace `-monto` y en `int()`). Primero cuántos billetes de 100 caben; lo que sobra se reparte en billetes de 50, y lo que sobra de eso, en billetes de 10.
</details>

---
## 🚀 Nivel pro (opcional)
1. `texto_miles`: `monto_grande` con el formato `"S/ 1,234,567.89"` (separador de miles y 2 decimales). Investiga cómo pedir el separador de miles en el formato de un f-string.
2. `texto_pct`: `tasa_morosidad` como porcentaje con 1 decimal, por ejemplo `"12.3%"`.
3. `codigo_invertido`: `codigo_venta` al revés. El slicing admite un tercer número, el paso: `texto[inicio:fin:paso]`. Investiga qué hace un paso negativo.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Ejecutar celdas, escribir comentarios y leer un error de abajo hacia arriba.
- [ ] Explicar la diferencia entre crear una variable y reasignarla.
- [ ] Decir el tipo de `7`, `7.0`, `"7"` y `True`, y convertir de uno a otro.
- [ ] Explicar por qué `"42" + "1"` no da `43`.
- [ ] Explicar la diferencia entre `/`, `//` y `%`, y qué pasa con `//` y `%` cuando el número es negativo.
- [ ] Usar paréntesis para que un cálculo con varios operadores dé lo que quiero.
- [ ] Sacar partes de un texto con índices positivos, negativos y slicing.
- [ ] Limpiar un texto con `strip`, `replace`, `upper` y `lower`, y cortarlo con `split`.
- [ ] Construir un mensaje con un f-string y números con 2 decimales.

**Próxima sesión (S02):** listas, tuplas, diccionarios, `if` y bucles.